In [ ]:
from google.colab import userdata

API_KEY = userdata.get("OPENROUTER_API_KEY")

if API_KEY:
    print("OpenRouter API key has been connected successfully!")
    print("Key:",API_KEY[:7]+"..."+API_KEY[-4:])
else:
    print("API key not found")

OpenRouter API key has been connected successfully!
Key: sk-or-v...2017


In [6]:
!pip install -q -U gradio google-genai

from google import genai
from google.colab import userdata
import gradio as gr
import re


# ============================================================
# 1. GEMMA API SETUP
# ============================================================

API_KEY = userdata.get("GEMMA_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMMA_API_KEY not found. "
        "Go to Colab Secrets and enable Notebook access."
    )

client = genai.Client(api_key=API_KEY)

MODEL = "gemma-4-31b-it"


# ============================================================
# 2. ALLERGEN KEYWORD MAP
# ============================================================

ALLERGEN_MAP = {
    "dairy": {
        "milk", "dairy", "paneer", "cheese",
        "butter", "cream", "curd", "yogurt",
        "kheer", "ghee"
    },
    "milk": {
        "milk", "dairy", "paneer", "cheese",
        "butter", "cream", "curd", "yogurt",
        "kheer", "ghee"
    },
    "peanut": {
        "peanut", "groundnut", "ground nut",
        "moongfali", "peanut butter"
    },
    "egg": {
        "egg", "eggs", "anda"
    },
    "soy": {
        "soy", "soya", "tofu"
    },
    "gluten": {
        "wheat", "maida", "atta",
        "bread", "roti", "paratha", "naan"
    }
}


def expand_allergies(allergy_text):
    words = set()

    for value in allergy_text.split(","):
        value = value.strip().lower()

        if not value:
            continue

        words.add(value)
        words.update(ALLERGEN_MAP.get(value, set()))

    return words


# ============================================================
# 3. CHEAP HOSTEL ALTERNATIVES
# ============================================================

ALTERNATIVES = [
    {
        "name": "Banana + Roasted Chana",
        "cost": 25,
        "reason": "Simple, cheap and requires no cooking."
    },
    {
        "name": "Aloo Sandwich",
        "cost": 40,
        "reason": "Bread and boiled potato can make a quick hostel-room meal."
    },
    {
        "name": "Peanut + Banana",
        "cost": 25,
        "reason": "Cheap snack option when peanut is not an allergy concern."
    },
    {
        "name": "Plain Rice + Ready-to-eat Dal",
        "cost": 45,
        "reason": "Simple backup meal when the mess menu is unsuitable."
    }
]


def choose_alternative(allergy_text):

    allergies = expand_allergies(allergy_text)

    for option in ALTERNATIVES:

        option_name = option["name"].lower()

        # Never suggest an alternative containing
        # an ingredient explicitly covered by the allergy input.
        blocked = any(
            word in option_name
            for word in allergies
        )

        if blocked:
            continue

        return option

    return {
        "name": "Plain rice + boiled potato",
        "cost": 30,
        "reason": "Simple backup option when the available menu does not fit the user's restrictions."
    }

# ============================================================
# 4. GEMMA ANALYSIS
# ============================================================

def get_gemma_analysis(menu, allergies, diet, budget):

    prompt = f"""
You are "Khana Kya Hai?", an AI assistant made for hostel students.

The user wants help understanding today's hostel mess menu.

MENU:
{menu}

ALLERGIES / INTOLERANCES:
{allergies}

DIETARY PREFERENCE:
{diet}

DAILY BUDGET:
{budget}

Your task is to analyze each menu item.

RULES:
1. Never give medical certainty.
2. If an item's ingredients or cooking method are unknown, put it under VERIFY.
3. If an item clearly contains a stated allergen, put it under AVOID.
4. Do not put the same item in AVOID and SUITABLE.
5. Do not recommend an item that appears in AVOID.
6. Do not invent exact ingredients when they are unknown.
7. Respect vegetarian / non-vegetarian / vegan preference.
8. Do not invent a price for hostel mess items.
9. The application will calculate alternative-food cost separately.

Return ONLY the following format:

STATUS: SAFE / CAUTION / HIGH_RISK

AVOID:
- item 1
- item 2

VERIFY:
- item 1
- item 2

SUITABLE:
- item 1
- item 2

MEAL:
one practical recommendation using ONLY items from SUITABLE

REASON:
short explanation

Always end with:
Verify ingredients, cooking medium and cross-contact with hostel mess staff.
"""

    try:

        response = client.models.generate_content(
            model=MODEL,
            contents=prompt
        )

        return response.text.strip()

    except Exception as error:
        return f"ERROR::{error}"


# ============================================================
# 5. PARSE GEMMA RESPONSE
# ============================================================

def parse_section(text, section_name, next_sections):

    pattern = rf"{section_name}:\s*(.*?)(?=\n(?:{'|'.join(next_sections)}):|\Z)"

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not match:
        return []

    lines = []

    for line in match.group(1).splitlines():

        line = line.strip()

        if line.startswith("-"):
            item = line[1:].strip()

            if item:
                lines.append(item)

    return lines


def parse_text_response(text):

    status_match = re.search(
        r"STATUS:\s*(SAFE|CAUTION|HIGH_RISK)",
        text,
        flags=re.IGNORECASE
    )

    status = (
        status_match.group(1).upper()
        if status_match
        else "CAUTION"
    )

    avoid_items = parse_section(
        text,
        "AVOID",
        ["VERIFY", "SUITABLE", "MEAL", "REASON"]
    )

    verify_items = parse_section(
        text,
        "VERIFY",
        ["SUITABLE", "MEAL", "REASON"]
    )

    suitable_items = parse_section(
        text,
        "SUITABLE",
        ["MEAL", "REASON"]
    )

    meal_match = re.search(
        r"MEAL:\s*(.*?)(?=\nREASON:|\Z)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    reason_match = re.search(
        r"REASON:\s*(.*)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    meal = (
        meal_match.group(1).strip()
        if meal_match
        else "No specific meal recommendation."
    )

    reason = (
        reason_match.group(1).strip()
        if reason_match
        else "Please verify the menu with mess staff."
    )

    return {
        "status": status,
        "avoid": avoid_items,
        "verify": verify_items,
        "suitable": suitable_items,
        "meal": meal,
        "reason": reason
    }


# ============================================================
# 6. DETERMINISTIC SAFETY / CONSISTENCY LAYER
# ============================================================

def apply_safety_rules(data, allergy_text):

    blocked_words = expand_allergies(allergy_text)

    final_avoid = []
    final_verify = []
    final_suitable = []

    # -------------------------
    # Clean AVOID
    # -------------------------

    for item in data["avoid"]:

        item_lower = item.lower()

        if item not in final_avoid:
            final_avoid.append(item)

    # -------------------------
    # Clean VERIFY
    # -------------------------

    for item in data["verify"]:

        if item in final_avoid:
            continue

        item_lower = item.lower()

        explicit_allergen = any(
            word in item_lower
            for word in blocked_words
        )

        if explicit_allergen:
            if item not in final_avoid:
                final_avoid.append(item)
        else:
            final_verify.append(item)

    # -------------------------
    # Clean SUITABLE
    # -------------------------

    for item in data["suitable"]:

        if item in final_avoid:
            continue

        item_lower = item.lower()

        contains_allergen = any(
            word in item_lower
            for word in blocked_words
        )

        if contains_allergen:
            if item not in final_avoid:
                final_avoid.append(item)
            continue

        final_suitable.append(item)

    # -------------------------
    # Fix contradictory MEAL
    # -------------------------

    meal_lower = data["meal"].lower()

    meal_unsafe = any(
        item.lower() in meal_lower
        for item in final_avoid
    )

    if meal_unsafe:

        if final_suitable:

            data["meal"] = (
                " + ".join(final_suitable[:2])
            )

        else:

            data["meal"] = (
                "No suitable-looking mess combination identified."
            )

    # -------------------------
    # Status correction
    # -------------------------

    if final_avoid and not final_suitable:

        data["status"] = "HIGH_RISK"

    elif final_verify and final_suitable:

        data["status"] = "CAUTION"

    elif final_suitable:

        data["status"] = "SAFE"

    else:

        data["status"] = "CAUTION"

    data["avoid"] = final_avoid
    data["verify"] = final_verify
    data["suitable"] = final_suitable

    return data


# ============================================================
# 7. MAIN APPLICATION FUNCTION
# ============================================================

def analyze_food(menu, allergies, diet, budget):

    if not menu or not menu.strip():

        return "## ⚠️ Please enter today's hostel mess menu."

    if not allergies or not allergies.strip():

        return "## ⚠️ Please enter at least one allergy/intolerance."

    gemma_result = get_gemma_analysis(
        menu,
        allergies,
        diet,
        budget
    )

    if gemma_result.startswith("ERROR::"):

        return (
            "## 🚨 Gemma Request Failed\n\n"
            "```text\n"
            + gemma_result[7:]
            + "\n```"
        )

    data = parse_text_response(
        gemma_result
    )

    data = apply_safety_rules(
        data,
        allergies
    )

    alternative = choose_alternative(
        allergies
    )

    # -------------------------
    # Status UI
    # -------------------------

    if data["status"] == "SAFE":

        status_display = "🟢 SAFE-LOOKING OPTIONS"

    elif data["status"] == "HIGH_RISK":

        status_display = "🚨 HIGH RISK / AVOID"

    else:

        status_display = "🟡 CAUTION / VERIFY"

    # -------------------------
    # Format lists
    # -------------------------

    avoid_text = "\n".join(
        f"- ❌ {item}"
        for item in data["avoid"]
    )

    verify_text = "\n".join(
        f"- ⚠️ {item}"
        for item in data["verify"]
    )

    suitable_text = "\n".join(
        f"- ✅ {item}"
        for item in data["suitable"]
    )

    if not avoid_text:
        avoid_text = "- None identified"

    if not verify_text:
        verify_text = "- None identified"

    if not suitable_text:
        suitable_text = "- None identified"

    return f"""
# 🍛 Khana Kya Hai?

## {status_display}

### ❌ Items to Avoid

{avoid_text}

### ⚠️ Items to Verify

{verify_text}

### ✅ Suitable-looking Items

{suitable_text}

### 🍽️ Recommended Meal

{data["meal"]}

### 🥪 Cheap Alternative

**{alternative["name"]}**

Estimated extra cost: **₹{alternative["cost"]}**

_{alternative["reason"]}_

### 💡 Why?

{data["reason"]}

---

### ⚠️ Safety Notice

This is an AI prototype, not medical advice.

Always verify ingredients, cooking medium and possible cross-contact with hostel mess staff.
"""


# ============================================================
# 8. GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="Khana Kya Hai?"
) as demo:

    gr.Markdown(
        """
# 🍛 KHANA KYA HAI?

### Ranchi Hostel Mess & Allergy Planner

**Powered by Google's Gemma open-weight model**

A simple assistant for hostel students dealing with mess menus,
food restrictions and everyday meal decisions.
"""
    )

    with gr.Row():

        with gr.Column():

            menu_input = gr.Textbox(
                label="🍽️ Today's Mess Menu",
                placeholder=(
                    "Example: Aloo Paratha, Paneer Butter Masala, "
                    "Rice, Dal, Kheer"
                ),
                lines=5
            )

            allergy_input = gr.Textbox(
                label="⚠️ Allergies / Intolerances",
                placeholder="Example: milk, dairy, peanut"
            )

            diet_input = gr.Dropdown(
                choices=[
                    "Vegetarian",
                    "Non-Vegetarian",
                    "Vegan"
                ],
                value="Vegetarian",
                label="🥗 Dietary Preference"
            )

            budget_input = gr.Textbox(
                label="💰 Daily Food Budget",
                value="₹100"
            )

            analyze_button = gr.Button(
                "🔍 Analyze My Food",
                variant="primary"
            )

        with gr.Column():

            output = gr.Markdown(
                """
## 👋 Ready?

Enter today's mess menu and your food restrictions.

Then click **Analyze My Food**.
"""
            )

    analyze_button.click(
        fn=analyze_food,
        inputs=[
            menu_input,
            allergy_input,
            diet_input,
            budget_input
        ],
        outputs=output
    )


# ============================================================
# 9. START APP
# ============================================================

demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://89e2f7646d24c04115.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
